In [4]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

DATA_PATH = Path("data/task_2_data_ex.csv")
KEY = ["plant_id", "year"]

## 1. Функции

In [5]:
INT_COLS = [
    "year", "month",
    "produced_material", "produced_material_production_type",
    "component_material", "component_material_production_type",
]
QTY_COLS = ["produced_material_quantity", "component_material_quantity"]


def load_bom(path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, dtype=str)
    return pd.read_csv(path, dtype=str)


def clean_bom(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c]).astype("Int64")
    for c in QTY_COLS:
        df[c] = pd.to_numeric(df[c].str.replace(",", "", regex=False))
    return df


def build_annual_edges(df: pd.DataFrame) -> pd.DataFrame:
    produced_year = (df.drop_duplicates(KEY + ["month", "produced_material"])
                       .groupby(KEY + ["produced_material"], as_index=False)["produced_material_quantity"].sum())

    attr_cols = ["produced_material_production_type", "produced_material_release_type",
                 "component_material_production_type", "component_material_release_type"]
    edges = (df.groupby(KEY + ["produced_material", "component_material"], as_index=False)
               .agg(**{c: (c, "first") for c in attr_cols},
                    component_material_quantity=("component_material_quantity", "sum")))
    return edges.merge(produced_year, on=KEY + ["produced_material"], how="left")


def explode_fin(edges: pd.DataFrame) -> pd.DataFrame:
    fin_lines = edges[(edges["produced_material_release_type"] == "FIN")
                      & (edges["component_material_release_type"] == "PROD")]
    frontier = (fin_lines[KEY + ["produced_material", "component_material"]]
                .rename(columns={"produced_material": "fin_material_id", "component_material": "current"}))

    frames, level = [], 0
    while not frontier.empty:
        step = frontier.merge(edges, left_on=KEY + ["current"], right_on=KEY + ["produced_material"], how="inner")
        step["level"] = level
        frames.append(step.drop(columns="current"))

        expandable = step["component_material_release_type"].isin(["PROD", "FIN"])
        frontier = (step.loc[expandable, KEY + ["fin_material_id", "component_material"]]
                    .drop_duplicates()
                    .rename(columns={"component_material": "current"}))
        level += 1

    return pd.concat(frames, ignore_index=True)


def format_result(exploded: pd.DataFrame, edges: pd.DataFrame) -> pd.DataFrame:
    fin_nodes = (edges[KEY + ["produced_material", "produced_material_release_type",
                              "produced_material_production_type", "produced_material_quantity"]]
                 .drop_duplicates()
                 .rename(columns={
                     "produced_material": "fin_material_id",
                     "produced_material_release_type": "fin_material_release_type",
                     "produced_material_production_type": "fin_material_production_type",
                     "produced_material_quantity": "fin_production_quantity"}))

    res = exploded.merge(fin_nodes, on=KEY + ["fin_material_id"], how="left")
    res = res.rename(columns={
        "plant_id": "plant",
        "produced_material": "prod_material_id",
        "produced_material_release_type": "prod_material_release_type",
        "produced_material_production_type": "prod_material_production_type",
        "produced_material_quantity": "prod_material_production_quantity",
        "component_material": "component_id",
        "component_material_quantity": "component_consumption_quantity"})

    res = res.sort_values(["plant", "fin_material_id", "year", "level", "prod_material_id", "component_id"])
    cols = ["plant", "fin_material_id", "fin_material_release_type", "fin_material_production_type",
            "fin_production_quantity", "prod_material_id", "prod_material_release_type",
            "prod_material_production_type", "prod_material_production_quantity", "component_id",
            "component_material_release_type", "component_material_production_type",
            "component_consumption_quantity", "year"]
    return res[cols].reset_index(drop=True)

## 2. Анализ данных

In [6]:
bom = clean_bom(load_bom(DATA_PATH))
print(bom.shape)
bom.head(12)

(1320, 11)


,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.0,50000,8002,PROD,990.0,RLT_10
1,2024,1,50000,8002,PROD,859.0,80070,8007,PROD,879.0,RLT_10
2,2024,1,50000,8002,PROD,859.0,90000,<NA>,ADD,50.0,RLT_10
3,2024,1,50000,8002,PROD,859.0,90001,<NA>,ADD,20.0,RLT_10
4,2024,1,80070,8007,PROD,929.0,80010,8001,PROD,3626.0,RLT_10
5,2024,1,80070,8007,PROD,929.0,90002,<NA>,ADD,30.0,RLT_10
6,2024,1,80070,8007,PROD,929.0,90003,<NA>,ADD,11.0,RLT_10
7,2024,1,80010,8001,PROD,1726.0,80000,8000,PROD,1818.0,RLT_10
8,2024,1,80010,8001,PROD,1726.0,90004,<NA>,ADD,101.0,RLT_10
9,2024,1,80000,8000,PROD,1980.0,70000,<NA>,RM,2668.0,RLT_10


In [ ]:
fin = bom[bom["produced_material_release_type"] == "FIN"]
(bom.groupby(["year", "plant_id"]).size().rename("rows").to_frame()
    .join(fin.groupby(["year", "plant_id"])["produced_material"].nunique().rename("fin_materials")))

year {np.int64(2024): 1320}
plant_id {'RLT_14': 792, 'RLT_10': 264, 'RLT_16': 264}
plant_id
RLT_10    2
RLT_14    6
RLT_16    2
Name: produced_material, dtype: int64


## 3. Результат

In [8]:
edges = build_annual_edges(bom)
result = format_result(explode_fin(edges), edges)
print(result.shape)
result

(100, 14)


,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007,11303.0,2024
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,<NA>,598.0,2024
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,<NA>,242.0,2024
3,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001,41769.0,2024
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,90002,ADD,<NA>,355.0,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,RLT_16,10008,FIN,8002,12067.0,80078,PROD,8007,10676.0,90043,ADD,<NA>,120.0,2024
96,RLT_16,10008,FIN,8002,12067.0,80018,PROD,8001,21410.0,80008,PROD,8000,23516.0,2024
97,RLT_16,10008,FIN,8002,12067.0,80018,PROD,8001,21410.0,90044,ADD,<NA>,1198.0,2024
98,RLT_16,10008,FIN,8002,12067.0,80008,PROD,8000,23397.0,70008,RM,<NA>,31048.0,2024
